In [7]:
pip install nltk scikit-learn scipy xgboost

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 3.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.1/318.1 MB 2.8 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
df_ok = pd.read_csv("output.csv")
print(len(df_ok))
df_ok = df_ok[df_ok['price_change_4h'].notna()]
df_ok = df_ok[df_ok['volume_surge_price'].notna()]
print(len(df_ok))
df_ok

20258
20251


,timestamp,title,source,volume_surge_price,price_change_4h
0,1629741303,Visa Makes Their Grand Entrance Into the NFT A...,Watcher Guru,-1.006608e-06,-0.000297
1,1629936472,Have you seen our billboard in Times Square to...,Watcher Guru,-2.814638e-03,-0.013606
2,1630005294,Record breaking sale for Bored Ape,Watcher Guru,6.691295e-03,0.013199
3,1630605534,"This is our only Telegram, please be careful o...",Watcher Guru,-9.948841e-07,0.000483
4,1631232301,Youtube Sensation MrBeast Reveals His Portfoli...,Watcher Guru,2.904352e-04,0.000779
...,...,...,...,...,...
20246,1743146885,NY AG files $200 million settlement with Galax...,unfolded.,-1.440233e-02,-0.009280
20247,1743169126,Terraform Labs to open crypto creditor claims ...,unfolded.,-7.996360e-02,-0.010471
20248,1743181914,FDIC clarifies that banks can engage in permit...,unfolded.,-2.301976e-02,-0.006169
20249,1743188025,Trump pardons three co-founders of BitMex cryp...,unfolded.,-2.509775e-04,0.000081


In [9]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /scratch/lustre/home/poma2368/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /scratch/lustre/home/poma2368/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /scratch/lustre/home/poma2368/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /scratch/lustre/home/poma2368/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import make_scorer
from scipy.stats import pearsonr
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Load Data
df = df_ok[['title', 'volume_surge_price', 'price_change_4h']][0:2000]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['cleaned_title'] = df['title'].apply(clean_text)

# Pearson correlation scorer
def pearson_corr(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]

corr_scorer = make_scorer(pearson_corr, greater_is_better=True)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('gbr', GradientBoostingRegressor())
])

# Define hyperparameter grid
#{'gbr__learning_rate': 0.1, 'gbr__max_depth': 3, 'gbr__n_estimators': 200, 'gbr__subsample': 1.0,
#'tfidf__max_df': 0.1, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 2), 'tfidf__sublinear_tf': False}

param_grid = {
    'tfidf__ngram_range': [(2,7),(2,5),(1,6),(3,3), (1,7), (3,4), (3,5)],
    'tfidf__max_df': [0.125],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100],
    'gbr__learning_rate': [0.07, 0.06],
    'gbr__max_depth': [4],
    'gbr__subsample': [1.0]
}

# Perform grid search
#grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring=corr_scorer, n_jobs=-1)
#grid_search.fit(df['cleaned_title'], df['volume_surge_price'])

# Output best parameters
#print(grid_search.best_params_)


```
param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2),(2,2),(2,3),(1,3)],
    'tfidf__max_df': [0.001,0.01,0.1,0.2],
    'tfidf__min_df': [1, 2],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100, 200],
    'gbr__learning_rate': [0.05, 0.1],
    'gbr__max_depth': [3, 5, 7],
    'gbr__subsample': [0.8, 1.0]
}
```
{'gbr__learning_rate': 0.1, 'gbr__max_depth': 3, 'gbr__n_estimators': 200, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.1, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 2), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(2,2),(2,4),(1,4)],
    'tfidf__max_df': [0.075,0.1,0.125],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100],
    'gbr__learning_rate': [0.08, 0.1],
    'gbr__max_depth': [2, 3, 4],
    'gbr__subsample': [0.95, 1.0]
}
```
{'gbr__learning_rate': 0.08, 'gbr__max_depth': 4, 'gbr__n_estimators': 100, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.125, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 2), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(2,2),(2,5),(1,5)],
    'tfidf__max_df': [0.175,0.15,0.125],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100],
    'gbr__learning_rate': [0.07, 0.08, 0.09],
    'gbr__max_depth': [4],
    'gbr__subsample': [1.1, 1.0]
}
```
{'gbr__learning_rate': 0.07, 'gbr__max_depth': 4, 'gbr__n_estimators': 100, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.125, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 5), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(2,7),(2,5),(1,6),(3,3), (1,7), (3,4), (3,5)],
    'tfidf__max_df': [0.125],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100],
    'gbr__learning_rate': [0.07, 0.06],
    'gbr__max_depth': [4],
    'gbr__subsample': [1.0]
}
```
{'gbr__learning_rate': 0.06, 'gbr__max_depth': 4, 'gbr__n_estimators': 100, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.125, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 7), 'tfidf__sublinear_tf': False}

In [13]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBRegressor
from scipy.stats import pearsonr
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from datetime import timedelta

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Data prep
df = df_ok[['title', 'volume_surge_price', 'price_change_4h', 'timestamp']]
df['cleaned_title'] = df['title'].apply(clean_text)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.sort_values('timestamp')

# Time-based split
cutoff_date = df['timestamp'].max() - timedelta(days=365)
train_df = df[df['timestamp'] < cutoff_date]
test_df = df[df['timestamp'] >= cutoff_date]

# TF-IDF
vectorizer = TfidfVectorizer(max_df=0.125, min_df=1, ngram_range=(2, 7), sublinear_tf=False)
x_train = vectorizer.fit_transform(train_df['cleaned_title'])
y_train = train_df['price_change_4h']

x_test = vectorizer.transform(test_df['cleaned_title'])
y_test = test_df['price_change_4h']

# Model
gbr = XGBRegressor(
    learning_rate=0.06,
    max_depth=4,
    subsample=1.0,
    n_estimators=1000,
    n_jobs=-1,
    verbosity=1
)
gbr.fit(x_train, y_train)
y_pred = gbr.predict(x_test)

In [14]:
true_price_change = test_df['price_change_4h']

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100
smape = 100 * np.mean(2.0 * np.abs(y_test - y_pred) / (np.abs(y_test) + np.abs(y_pred) + 1e-8))
r2 = r2_score(y_test, y_pred)

corr, corr_p = pearsonr(y_pred, true_price_change)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"SMAPE: {smape:.2f}%")
print(f"R²: {r2:.4f}")
print(f"Correlation with price_change_4h: {corr:.4f}")
print(f"Correlation p: {corr_p:.4f}")

MSE: 0.0003
RMSE: 0.0159
MAE: 0.0111
MAPE: 3249.21%
SMAPE: 183.26%
R²: -0.0376
Correlation with price_change_4h: 0.0543
Correlation p: 0.0049


MSE: 0.0034
RMSE: 0.0585
MAE: 0.0203
MAPE: 8109.95%
SMAPE: 156.27%
R²: -0.0543
Correlation with price_change_4h: 0.1713


In [ ]:
pip install joblib

In [ ]:
import joblib

joblib.dump(gbr, 'gbr_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
